>[Naive RAG vs GraphRAG with Neo4J & Weaviate](#scrollTo=n3QFDMrgAkCo)

>>[Install Dependencies](#scrollTo=n3QFDMrgAkCo)

>>[Write Documents to Weaviate Cloud](#scrollTo=nqwuGr0Xhgtm)

>>[Classic RAG with OpenAI](#scrollTo=-uAAWPQXBUdX)

>>[Graph RAG](#scrollTo=zzBnUF4bBYKG)

>>>[Build a Graph with Neo4J](#scrollTo=zzBnUF4bBYKG)

>>>[Extract Relevant Entities](#scrollTo=FVzpKJViBiJT)

>>>[Summarize Nodes and Communities](#scrollTo=j1wAsUfIBrGc)

>>>[Write the Entities to Weaviate](#scrollTo=n105cc-_B9bN)



## Naive RAG vs GraphRAG with Neo4J & Weaviate

In this recipe, we will be walking through 2 ways of doing RAG:
1. Classic RAG where we do simple vector search, followed be answer generation based on this context
2. Graph RAG, making use of both vector search, combined by a graph representation of our dataset including community and node summaries

For this example, we will be using a generated dataset called "Financial Contracts", that lists (fake) contracts sugned between individuals and companies.

## Write Documents to Weaviate Cloud

To get started, you can use a free Weaviate Sandbox.

1. Create a cluster
2. Take note of the cluster URL and API key
3. Go to 'Embeddings' and turn it on.

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

# Load credentials before importing datasets or other Hugging Face clients.
load_dotenv()

if not os.environ.get("HF_TOKEN"):
    hf_token = getpass("Hugging Face read token (Enter to use public downloads without authentication): ").strip()
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
    del hf_token

if "WEAVIATE_API_KEY" not in os.environ:
  os.environ["WEAVIATE_API_KEY"] = getpass("Weaviate API Key")
if "WEAVIATE_URL" not in os.environ:
  os.environ["WEAVIATE_URL"] = getpass("Weaviate URL")

In [2]:
import weaviate
from weaviate.auth import Auth

weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ.get("WEAVIATE_URL"),
    auth_credentials=Auth.api_key(os.environ.get("WEAVIATE_API_KEY")),
)

In [3]:
from weaviate.classes.config import Configure

# Your Weaviate instance is capped at 1 collection total, so clear out whichever
# one survived from a previous run (Financial_contracts or Entities) before creating.
weaviate_client.collections.delete("Financial_contracts")
weaviate_client.collections.delete("Entities")
weaviate_client.collections.create(
    "Financial_contracts",
    description="A dataset of financial contracts between indivicuals and/or companies, as well as information on the type of contract and who has authored them.",
    vector_config=Configure.Vectors.text2vec_weaviate(),
)

In [2]:
from datasets import load_dataset

financial_dataset = load_dataset("weaviate/agents", "query-agent-financial-contracts", split="train", streaming=True)

In [5]:
financial_collection = weaviate_client.collections.get("Financial_contracts")

with financial_collection.batch.dynamic() as batch:
    for item in financial_dataset:
        batch.add_object(properties=item["properties"])

## Classic RAG with OpenAI

In [6]:
from openai import AsyncOpenAI

client = AsyncOpenAI(
    base_url=os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
    api_key="ollama",
)

async def achat(messages, model="gpt-oss:20b", temperature=0, config={}):
    response = await client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=messages,
        **config,
    )
    return response.choices[0].message.content

In [7]:
async def classic_rag(input: str) -> str:
    context = [str(obj.properties) for obj in financial_collection.query.near_text(query = input, limit=3).objects]
    messages = [
    {
        "role": "user",
        "content": "Based on the given context: {context} \n\n Answer the following question: {question}".format(context=context, question=input)
    },
    ]
    output = await achat(messages, model="gpt-oss:20b")
    return output

In [8]:
response = await classic_rag("What do you know about Weaviate")
print(response)

**Weaviate – key facts from the documents**

| Item | Details |
|------|---------|
| **Legal status** | Corporation (organized under Delaware law in the first agreement; under California law in the second and third agreements). |
| **Principal office** | 123 Innovation Drive (location varies by agreement: Wilmington, DE; Tech City; San Francisco, CA). |
| **Partnership role** | Collaborates with OpenAI on AI‑related projects. |
| **Financial contributions** | • 15 Mar 2022: $481.42 (first agreement).  <br>• 5 Apr 2023: $340.31 (second agreement).  <br>• 15 Nov 2023: $244.46 (third agreement). |
| **Other financial terms** | In the second agreement, Weaviate shares additional operational and marketing costs equally with OpenAI. |
| **Responsibilities** | • Project management and client relations (second agreement).  <br>• Marketing and promotion of joint projects (third agreement).  <br>• Shared responsibilities for research, development, and technical support (first agreement). |
| **D

## Graph RAG

### Build a Graph with Neo4J


In [10]:
import os
from getpass import getpass
from my_graphrag import KBGraphRAG
from neo4j import GraphDatabase
import pandas as pd

In [11]:
driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
    #notifications_min_severity="OFF",
)
ms_graph = KBGraphRAG(driver=driver, database="financial-contracts", model="ollama/gpt-oss:20b", max_workers=2)

In [12]:
import pandas as pd

# Uses HF_TOKEN loaded in the initial setup cell.
df = pd.read_parquet("hf://datasets/weaviate/agents/query-agent/financial-contracts/0001.parquet")
df.head()

,properties,vector
0,"{'date': '2023-03-15T14:30:00+00:00', 'contrac...","[-0.02571106, 0.028182983, 0.011955261, -0.014..."
1,"{'date': '2023-03-15T10:30:00+00:00', 'contrac...","[0.013244629, 0.015045166, 0.0042381287, -0.04..."
2,"{'date': '2022-03-15T09:30:00+00:00', 'contrac...","[-0.041625977, 0.013809204, -0.011222839, 0.03..."
3,"{'date': '2023-09-15T14:23:00+00:00', 'contrac...","[-0.006462097, -0.003780365, -0.064453125, -0...."
4,"{'date': '2024-03-15T10:30:00+00:00', 'contrac...","[-0.022064209, 0.04623413, -0.0284729, 0.00141..."


In [13]:
texts = [el['contract_text'] for el in df['properties']]
texts[:2]

['PARTNERSHIP AGREEMENT\n\nThis Partnership Agreement ("Agreement") is made and entered into as of the 15th day of March, 2023, by and between Weaviate, a company registered in the State of California, and OpenAI, a research organization based in San Francisco, California.\n\n1. Purpose\nThe parties agree to establish a partnership to collaborate on artificial intelligence research and development, sharing resources and expertise.\n\n2. Contributions\nWeaviate shall contribute technology resources valued at $112.85 and staff time equivalent to a monetary value of $550.09. OpenAI shall contribute its research expertise and a project management team valued at $98.14.\n\n3. Profit Sharing\nThe net profits generated from joint projects shall be distributed as follows: Weaviate shall receive 60% and OpenAI shall receive 40%.\n\n4. Duration\nThis Agreement shall commence on the date hereof and shall continue in effect for a period of three (3) years, unless terminated earlier in accordance w

### Extract Relevant Entities

Next, we will start extracting relevant entities and relations between these entities that we might be interested in.

In [17]:
allowed_entities = ["Person", "Organization", "Location"]

await ms_graph.extract_nodes_and_rels(texts, allowed_entities)

Extracting nodes & relationships: 100%|██████████| 100/100 [00:34<00:00,  2.87it/s]


'Successfuly extracted and imported 274 relationships'

### Summarize Nodes and Communities

In [18]:
await ms_graph.summarize_nodes_and_rels()

Summarizing relationships: 100%|██████████| 33/33 [00:08<00:00,  3.87it/s]


'Successfuly summarized nodes and relationships'

In [21]:
await ms_graph.summarize_communities()

Leiden algorithm identified 1 community levels with 3 communities on the last level.




Summarizing communities:   0%|          | 0/3 [00:00<?, ?it/s]

Summarizing communities:  33%|███▎      | 1/3 [00:09<00:18,  9.19s/it]

Summarizing communities:  67%|██████▋   | 2/3 [00:10<00:04,  4.57s/it]

Summarizing communities: 100%|██████████| 3/3 [00:13<00:00,  4.41s/it]


'Generated 3 community summaries'

In [14]:
entities = ms_graph.query("""
MATCH (e:__Entity__)
RETURN e.name AS entity_id, e.summary AS entity_summary
""")

In [15]:
entities[:2]

[{'entity_id': 'MARK ROBSON',
  'entity_summary': 'Mark\u202fRobson is a multifaceted individual whose professional and contractual activities span several organisations and roles. Primarily, he is a Software Engineer employed by Weaviate, where he contributes to software development and support services. In addition to his engineering duties, Robson acts as the principal point of contact for Weaviate in its partnership with OpenAI, where he frequently serves as the authorized representative and signatory of purchase orders, partnership agreements, and sales agreements on behalf of the company.\n\nRobson’s involvement does not end with corporate agreements. He has entered into a number of client‑ and service‑related contracts with Weaviate, acting as buyer of software licences and services, a lessee of commercial premises, and a borrower in several loan arrangements. His contractual footprint also includes positions such as the recipient of invoices and the signing party on a variety o

### Write the Entities to Weaviate

In [16]:
from weaviate.classes.config import Configure

# Your Weaviate instance is capped at 1 collection, so Financial_contracts (only
# needed for the earlier Classic RAG demo) has to go before Entities can be created.
weaviate_client.collections.delete("Financial_contracts")
weaviate_client.collections.create(
    "Entities",
    description="A dataset of entities appearing in the financial contracts between indivicuals and/or companies, as well as information on the type of contract and who has authored them.",
    vector_config=Configure.Vectors.text2vec_weaviate(),
)

In [17]:
from datasets import IterableDataset

# Define a simple generator
def list_generator(data):
    for item in data:
        yield item

# Create the IterableDataset
entities_dataset = IterableDataset.from_generator(list_generator, gen_kwargs={"data": entities})

In [18]:
entities_collection = weaviate_client.collections.get("Entities")

with entities_collection.batch.dynamic() as batch:
    for item in entities_dataset:
        batch.add_object(properties=item)

In [19]:
from neo4j_graphrag.retrievers import WeaviateNeo4jRetriever

retrieval_query = """
    WITH collect(node) as nodes
WITH collect {
    UNWIND nodes as n
    MATCH (n)<-[:MENTIONS]->(c:__Chunk__)
    WITH c, count(distinct n) as freq
    RETURN c.text AS chunkText
    ORDER BY freq DESC
    LIMIT 3
} AS text_mapping,
collect {
    UNWIND nodes as n
    MATCH (n)-[:IN_COMMUNITY*]->(c:__Community__)
    WHERE c.summary IS NOT NULL
    WITH c, c.rating as rank
    RETURN c.summary
    ORDER BY rank DESC
    LIMIT 3
} AS report_mapping,
collect {
    UNWIND nodes as n
    MATCH (n)-[r:SUMMARIZED_RELATIONSHIP]-(m)
    WHERE m IN nodes
    RETURN r.summary AS descriptionText
    LIMIT 3
} as insideRels,
collect {
    UNWIND nodes as n
    RETURN n.summary AS descriptionText
} as entities
RETURN {Chunks: text_mapping, Reports: report_mapping,
       Relationships: insideRels,
       Entities: entities} AS output
    """

retriever = WeaviateNeo4jRetriever(
    driver=driver,
    client=weaviate_client,
    collection="Entities",
    id_property_external="entity_id",
    id_property_neo4j="name",
    retrieval_query=retrieval_query
)

In [20]:
async def hybrid_local_search_rag(input: str) -> str:
    context = [str(el[1]) for el in retriever.search(query_text=input, top_k=3)]
    messages = [
    {
        "role": "user",
        "content": "Based on the given context: {context} \n\n Answer the following question: {question}".format(context=context, question=input)
    },
    ]
    output = await achat(messages, model="gpt-oss:20b")
    return output

In [21]:
response = await hybrid_local_search_rag(input="What do you know about Weaviate")
print(response)

**Weaviate** is an open‑source, cloud‑native vector search engine that doubles as a lightweight knowledge‑graph database.  It lets you store “objects” (records) that have both structured properties and a high‑dimensional vector representation (typically an embedding produced by a language model).  The engine then lets you perform semantic search, similarity queries, and graph‑based traversals all in one place.

Key points you’ll find useful:

| Feature | What it means |
|---------|---------------|
| **Vector index** | Uses HNSW (Hierarchical Navigable Small World) for fast approximate nearest‑neighbour search. |
| **Schema‑driven** | You define a schema (classes, properties, data types) and Weaviate validates and stores objects accordingly. |
| **Graph capabilities** | Objects can be linked via relationships, giving you a simple graph layer on top of the vector index. |
| **APIs** | Exposes both a GraphQL API (for flexible queries) and a REST API (for simple CRUD). |
| **Embeddings** |